# 耗时差异分析 (Timing Discrepancy Analysis)

在 `summary.csv` 中记录的 `训练耗时` 和 `推理耗时` 并没有包含整个程序的全部生命周期。具体来说，以下几个主要环节由于写在计时器 `t0` 之前或由于是在主循环外执行，导致了实际等待时间比日志记录的纯训练/推理耗时要长得多：

### 1. 缺失掩码模拟 (Missingness Simulation)
在 `src/experiments/runner.py` 中，每遍历一个参数组合时都会调用 `build_missing_inputs()`。该函数内部调用 `simulate_missingness` 来动态生成 train/val/test 三个子集的掩码(Mask)。如果是简单随机 `mcar` 还好，但在处理 `seq` (连续块缺失) 和 `scm` (空间相关缺失) 时，相关的采样和屏蔽逻辑（如聚类处理）较为耗时。

### 2. 重复的数据加载与内存拷贝 (Repeated Data Loading & Memory Copy)
在 `src/models/mymodel/runner.py` 的 `run_mymodel_on_splits` 中，执行了 `ground_X, _, all_stations, _, vars_info = load_data()`。这意味着外层 `runner.py` 已经加载过一次数据，而在进入具体模型 run 函数时又发生了一次重复读取。随后还需要对大矩阵进行 `transpose` 操作和按需时间段进行切片 (`X_splits[split_name]`)，触发了消耗内存与CPU的大规模张量复制。

### 3. DataLoader 大规模张量构建
调用 `build_spatial_dataloader()` 时，使用了把二维数据变换为三维窗口数据的 `prepare_spatial_windows()`。该方法采用了 `reshape` + `transpose`，然后再强转为 `float32` 并封装进 `torch.tensor` 和 `TensorDataset` 中。这些大张量的分配与创建也在计时器 `t0` 开始之前发生。

### 4. 模型初始化与显存搬迁
`STAT_Model(...)` 对象的实例化，以及随后的 `.to(device)` 操作（将几兆到几千万的模型权重拷贝至 MPS 显存）消耗了一定的时间。这也是发生在计时之前。

### 5. 垃圾回收与缓存清理 (GC & Empty Cache)
在每次跑完模型后、验证结束后、进行 HPO 的间隔等位置，代码中显式调用了 `gc.collect()` 以及 `torch.mps.empty_cache()`。在 macOS 上清空 MPS 显存具有相对较高的延迟开销，它属于系统底层的同步操作，这积攒下来严重拖慢了每一轮整体外层循环的流转。

### 6. 指标评估与 HPO Overhead
预测完成后使用 numpy 算一遍 `rmse_on_missing_2d`，以及在开启超参搜索(`hpo_trials > 0`)时 optuna 的创建、调度、记录也是额外耗掉的时间（虽然针对目标更新了记录逻辑，但整体额外开销仍在）。

## 针对性优化方案 (Optimizations Implemented)

为了减少上述“隐藏时间”并加快整体实验跑批速度，同时防止内存爆满导致系统崩溃，在代码层面进行了以下关键优化：

### 1. 将全量预计算掩码改为迭代生成 (Iterative Mask Generation)
之前为了加速计算，在主循环外一次性预计算了所有 (pattern, pi) 组合的掩码。但在大批量实验（如 3 模式 x 5 比例 = 15 组掩码）时，这会瞬间吃掉数 GB 内存。
**优化后**：掩码生成逻辑被重新移入循环内部（针对每个 pattern 和 pi 实时生成）。每跑完一个组合下的所有模型测试，立即显式 `del` 掩码对象并调用 `gc.collect()`，确保内存及时归还系统。

### 2. 激进的显存与内存清理 (Aggressive Memory Management)
在 `mymodel` 专用 runner 和全局 `dispatcher` 中增加了更生硬的“预测后销毁”逻辑：
- 在获取到最终的 numpy 格式指标后，立即 `del model, loaders, X_win_splits`。
- 随后强制执行 `gc.collect()` 和 `torch.mps.empty_cache()`，这在 Mac MPS 系统上尤为重要，能有效防止因显存碎片堆积导致的系统卡顿或崩毁。

### 3. DataLoader 预处理优化与数据透传
消除了 `mymodel` 内部重复加载全量数据的开销。现在主程序会在启动瞬间完成 `ground_X` 的预加载与切分，随后以只读引用的方式透传给各个模型。这不仅减少了 I/O 耗时，更大幅降低了峰值内存占用。